# U11 — 주동작 원판 초기 후보 자동 선택

Dataset V2 detector가 만든 첫 후보 frame에서 주동작 원판을 자동 선택하고 수동 target과 비교합니다.

## 1. 목표와 범위

- 후보가 하나면 자동 선택합니다.
- 후보가 여러 개면 주동작자 중심과의 거리가 명확히 가까운 후보만 선택합니다.
- 거리가 비슷하거나 Pose 중심이 없으면 억지로 고르지 않고 `confirmation_required`로 남깁니다.
- 새 모델을 학습하지 않고 기존 detector·Pose·tracker를 재사용합니다.
- Task 1은 A01/B01/C01, Task 2는 겹치지 않는 A/B/C 각 3명을 검증합니다.

In [ ]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snatch_analysis.config import POSE_MODEL_NAME
from snatch_analysis.pipeline import save_json_atomically
from snatch_analysis.plate import (
    decide_initial_plate_selection_evaluation,
    evaluate_initial_plate_selections,
    infer_plate_video_candidates,
    save_plate_candidate_cache,
    save_plate_track,
    select_initial_plate_candidate,
    select_primary_plate_track,
    validate_initial_plate_targets,
    validate_u11_task2_video_selection,
)
from snatch_analysis.pose import (
    infer_pose_video_candidates,
    resolve_inference_device,
    save_pose_candidate_cache,
    save_selected_pose_keypoints,
    select_initial_primary_lifter,
    select_primary_lifter_track,
    verify_ultralytics_safe_load,
)
from snatch_analysis.video import read_video_frame

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

## 2. 실험 설정

`U11_AMBIGUITY_MARGIN`은 가장 가까운 후보와 두 번째 후보의 상대 거리 차이가 어느 정도여야 자동 선택할지 정합니다. 기본 0.10은 거리 차이가 10% 이하면 사용자 확인을 요청한다는 뜻입니다. 다른 값은 이번 기준선 결과를 본 뒤 별도 실험에서만 바꿉니다.

In [ ]:
RUN_U11_AUTO_SELECTION = False
U11_AMBIGUITY_MARGIN = 0.10
U11_VIDEO_IDS = ('a01_sn_f_01', 'b01_sn_f_01', 'c01_sn_f_01')

EVALUATION_SPLIT_PATH = PROJECT_ROOT / 'data/splits/evaluation_videos.csv'
U9_POSE_ROOT = PROJECT_ROOT / 'artifacts/u9_baseline_conf010'
U10_EVALUATION_ROOT = PROJECT_ROOT / 'artifacts/u10_plate_dataset_v2/evaluation'
U11_ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection'
U11_SELECTION_PATH = U11_ARTIFACT_ROOT / 'initial_selections.csv'
U11_SUMMARY_PATH = U11_ARTIFACT_ROOT / 'run_summary.json'
U11_CONTACT_SHEET_PATH = U11_ARTIFACT_ROOT / 'initial_selection_contact_sheet.png'

if not 0 <= U11_AMBIGUITY_MARGIN <= 1:
    raise ValueError('U11_AMBIGUITY_MARGIN은 0 이상 1 이하여야 합니다.')

## 3. 저장된 후보·Pose·수동 기준 확인

U10에서 만든 raw 후보와 수동 초기 선택, U9에서 저장한 주동작자 bbox 중심을 읽습니다. 모델을 다시 추론하지 않습니다.

In [ ]:
evaluation_split = pd.read_csv(EVALUATION_SPLIT_PATH)
pilot_rows = (
    evaluation_split.loc[evaluation_split['video_id'].isin(U11_VIDEO_IDS)]
    .set_index('video_id')
    .loc[list(U11_VIDEO_IDS)]
    .reset_index()
)
if len(pilot_rows) != len(U11_VIDEO_IDS):
    raise ValueError('U11 pilot 영상 metadata가 모두 필요합니다.')
if set(pilot_rows['read_status']) != {'ok'}:
    raise ValueError('읽기 오류가 있는 pilot 영상은 사용할 수 없습니다.')

required_paths = []
for video_id in U11_VIDEO_IDS:
    required_paths.extend([
        U10_EVALUATION_ROOT / video_id / 'plate_candidates.csv',
        U10_EVALUATION_ROOT / video_id / 'run_summary.json',
        U9_POSE_ROOT / video_id / 'pose_keypoints.csv',
    ])
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('U11 입력 artifact가 없습니다:\n' + '\n'.join(missing_paths))
print(f'U11 입력 확인: {len(U11_VIDEO_IDS)}영상, 파일 {len(required_paths)}개')

## 4. 초기 후보 자동 선택과 tracker 연결

`RUN_U11_AUTO_SELECTION=True`로 바꾸면 자동 선택 결과를 기존 U10 수동 선택과 비교하고, 선택 성공 영상만 기존 tracker에 전달합니다.

In [ ]:
u11_selections = pd.DataFrame()
if not RUN_U11_AUTO_SELECTION:
    print('U11 실행 대기: RUN_U11_AUTO_SELECTION=True로 바꾸세요.')
else:
    U11_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    for stale_path in (U11_SELECTION_PATH, U11_SUMMARY_PATH, U11_CONTACT_SHEET_PATH):
        stale_path.unlink(missing_ok=True)
    selection_rows = []
    total_frame_count = 0
    track_evaluated_frame_count = 0
    total_ok_frames = 0
    metadata_by_video = pilot_rows.set_index('video_id')
    for video_id in U11_VIDEO_IDS:
        metadata = metadata_by_video.loc[video_id]
        candidates = pd.read_csv(
            U10_EVALUATION_ROOT / video_id / 'plate_candidates.csv'
        )
        pose_keypoints = pd.read_csv(
            U9_POSE_ROOT / video_id / 'pose_keypoints.csv',
            usecols=['frame_index', 'bbox_center_x', 'bbox_center_y'],
        )
        lifter_centers = (
            pose_keypoints[['frame_index', 'bbox_center_x', 'bbox_center_y']]
            .dropna()
            .drop_duplicates('frame_index')
            .rename(columns={
                'bbox_center_x': 'center_x_px',
                'bbox_center_y': 'center_y_px',
            })
        )
        selection = select_initial_plate_candidate(
            candidates,
            lifter_centers=lifter_centers,
            ambiguity_margin=U11_AMBIGUITY_MARGIN,
        )
        manual_summary = json.loads(
            (U10_EVALUATION_ROOT / video_id / 'run_summary.json').read_text(
                encoding='utf-8'
            )
        )['plate']
        manual_frame_value = manual_summary.get('initial_frame_index')
        manual_candidate_value = manual_summary.get('initial_candidate_id')
        manual_seed_available = (
            manual_frame_value is not None and not pd.isna(manual_frame_value)
            and manual_candidate_value is not None and not pd.isna(manual_candidate_value)
        )
        manual_frame_index = int(manual_frame_value) if manual_seed_available else None
        manual_candidate_id = int(manual_candidate_value) if manual_seed_available else None
        selection.update({
            'video_id': video_id,
            'group': str(metadata['group']),
            'manual_frame_index': manual_frame_index,
            'manual_candidate_id': manual_candidate_id,
            'matches_manual': (
                selection['status'] == 'selected'
                and selection['frame_index'] == manual_frame_index
                and selection['candidate_id'] == manual_candidate_id
            ) if manual_seed_available else None,
        })
        track_path = U11_ARTIFACT_ROOT / video_id / 'plate_track.csv'
        track_path.unlink(missing_ok=True)
        total_frame_count += int(metadata['frame_count'])
        if selection['status'] == 'selected':
            track = select_primary_plate_track(
                candidates,
                frame_count=int(metadata['frame_count']),
                initial_frame_index=int(selection['frame_index']),
                initial_candidate_id=int(selection['candidate_id']),
                lifter_centers=lifter_centers,
                fps=float(metadata['fps']),
            )
            save_plate_track(track, track_path)
            selection['track_ok_frame_count'] = int(track['status'].eq('ok').sum())
            track_evaluated_frame_count += int(metadata['frame_count'])
            total_ok_frames += int(selection['track_ok_frame_count'])
        else:
            selection['track_ok_frame_count'] = None
        selection_rows.append(selection)

    u11_selections = pd.DataFrame(selection_rows)
    u11_selections.to_csv(U11_SELECTION_PATH, index=False)
    selected_count = int(u11_selections['status'].eq('selected').sum())
    manual_comparison_mask = u11_selections['matches_manual'].notna()
    manual_comparison_count = int(manual_comparison_mask.sum())
    manual_match_count = int(u11_selections['matches_manual'].eq(True).sum())
    wrong_auto_selection_count = int((
        u11_selections['status'].eq('selected')
        & manual_comparison_mask
        & ~u11_selections['matches_manual'].eq(True)
    ).sum())
    confirmation_count = int(
        u11_selections['status'].eq('confirmation_required').sum()
    )
    not_available_count = int(u11_selections['status'].eq('not_available').sum())
    u11_summary = {
        'status': 'complete',
        'video_count': len(u11_selections),
        'selected_count': selected_count,
        'manual_comparison_count': manual_comparison_count,
        'manual_match_count': manual_match_count,
        'wrong_auto_selection_count': wrong_auto_selection_count,
        'confirmation_required_count': confirmation_count,
        'not_available_count': not_available_count,
        'manual_review_rate': (confirmation_count + not_available_count) / len(u11_selections),
        'track_ok_frame_count': total_ok_frames,
        'track_evaluated_frame_count': track_evaluated_frame_count,
        'track_not_evaluated_frame_count': total_frame_count - track_evaluated_frame_count,
        'total_frame_count': total_frame_count,
        'track_coverage': (
            total_ok_frames / track_evaluated_frame_count
            if track_evaluated_frame_count else None
        ),
        'ambiguity_margin': U11_AMBIGUITY_MARGIN,
    }
    save_json_atomically(u11_summary, U11_SUMMARY_PATH)
    display(u11_selections[[
        'video_id', 'group', 'status', 'reason', 'frame_index', 'candidate_id',
        'candidate_count', 'distance_margin', 'matches_manual',
        'track_ok_frame_count',
    ]])
    display(pd.DataFrame([u11_summary]))

## 5. 첫 후보 frame 시각 검수

초록색은 자동 선택, 주황색은 선택하지 않은 후보, 빨간색은 확인이 필요한 후보입니다. 표의 `matches_manual`과 그림을 함께 확인합니다.

In [ ]:
if u11_selections.empty and U11_SELECTION_PATH.exists():
    u11_selections = pd.read_csv(U11_SELECTION_PATH)
if u11_selections.empty:
    print('시각 검수 대기: 먼저 U11 자동 선택을 실행하세요.')
else:
    metadata_by_video = pilot_rows.set_index('video_id')
    figure, axes = plt.subplots(1, len(u11_selections), figsize=(18, 7), squeeze=False)
    for axis, selection in zip(axes.flat, u11_selections.itertuples(index=False), strict=True):
        if pd.isna(selection.frame_index):
            axis.set_title(f'{selection.video_id} | {selection.status}')
            axis.axis('off')
            continue
        metadata = metadata_by_video.loc[selection.video_id]
        candidates = pd.read_csv(
            U10_EVALUATION_ROOT / selection.video_id / 'plate_candidates.csv'
        )
        first_candidates = candidates.loc[
            candidates['frame_index'].eq(int(selection.frame_index))
        ]
        frame = read_video_frame(
            PROJECT_ROOT / str(metadata['path']), int(selection.frame_index)
        ).copy()
        for candidate in first_candidates.itertuples(index=False):
            if selection.status == 'confirmation_required':
                color = (0, 0, 255)
            elif int(candidate.candidate_id) == int(selection.candidate_id):
                color = (0, 255, 0)
            else:
                color = (0, 165, 255)
            start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
            end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
            cv2.rectangle(frame, start, end, color, 4)
            cv2.putText(
                frame, f'id={int(candidate.candidate_id)}', start,
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2,
            )
        axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axis.set_title(
            f'{selection.video_id} | {selection.status} | manual_match={selection.matches_manual}'
        )
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(U11_CONTACT_SHEET_PATH, dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    print(f'검수 그림: {U11_CONTACT_SHEET_PATH}')

## 6. Task 1 결론과 Task 2 시작

Task 1은 A01/B01/C01에서 수동 seed와 3/3 일치했지만 모두 단일 후보였습니다. Task 2는 이전 detector 데이터와 사람이 겹치지 않는 A/B/C 각 3영상을 추가해 실제 다중 후보와 안전한 보류를 확인합니다. 새 모델이나 복합 점수는 추가하지 않습니다.

In [ ]:
RUN_U11_TASK2_INFERENCE = False
RUN_U11_TASK2_EVALUATION = False
U11_TASK2_VIDEO_IDS = (
    'a02_sn_f_01', 'a06_sn_h_01', 'a08_sn_f_01',
    'b02_sn_f_01', 'b04_sn_f_01', 'b05_sn_f_01',
    'c03_sn_f_01', 'c04_sn_f_01', 'c05_sn_f_01',
)
U11_TASK2_REQUESTED_DEVICE = 'mps'
U11_TASK2_POSE_IMAGE_SIZE = 640
U11_TASK2_POSE_CONFIDENCE = 0.25
U11_TASK2_KEYPOINT_CONFIDENCE = 0.50
U11_TASK2_PLATE_IMAGE_SIZE = 320
U11_TASK2_PLATE_CONFIDENCE = 0.10

# 후보 contact sheet를 본 뒤 (first_frame_index, candidate_id),
# 'target_missing', 'no_plate' 중 하나로 바꿉니다.
U11_TASK2_MANUAL_TARGETS = {
    video_id: None for video_id in U11_TASK2_VIDEO_IDS
}

U11_TASK2_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection/task2'
U11_TASK2_INPUT_ROOT = U11_TASK2_ROOT / 'inputs'
U11_TASK2_CANDIDATE_REVIEW_PATH = U11_TASK2_ROOT / 'candidate_review.png'
U11_TASK2_SELECTION_REVIEW_PATH = U11_TASK2_ROOT / 'selection_review.png'
U11_TASK2_TARGET_PATH = U11_TASK2_ROOT / 'manual_targets.csv'
U11_TASK2_EVALUATION_PATH = U11_TASK2_ROOT / 'selection_evaluation.csv'
U11_TASK2_GROUP_SUMMARY_PATH = U11_TASK2_ROOT / 'group_summary.csv'
U11_TASK2_SUMMARY_PATH = U11_TASK2_ROOT / 'run_summary.json'
U11_TASK2_MANIFEST_PATH = PROJECT_ROOT / 'data/metadata/videos.csv'
U11_TASK2_U10_SPLIT_PATH = PROJECT_ROOT / 'data/plate_dataset_v2/video_split.csv'
U11_TASK2_WEIGHTS_PATH = (
    PROJECT_ROOT / 'artifacts/u10_plate_dataset_v2/training/weights/best.pt'
)

def validate_task2_reused_summary(
    summary, *, video_id, frame_count, weights_sha256
):
    pose = summary.get('pose', {})
    plate = summary.get('plate', {})
    expected = {
        'video_id': video_id,
        'frame_count': int(frame_count),
        'pose.status': 'complete',
        'pose.model': POSE_MODEL_NAME,
        'pose.image_size': U11_TASK2_POSE_IMAGE_SIZE,
        'pose.confidence': U11_TASK2_POSE_CONFIDENCE,
        'pose.keypoint_confidence': U11_TASK2_KEYPOINT_CONFIDENCE,
        'pose.processed_frame_count': int(frame_count),
        'plate.status': 'complete',
        'plate.model': str(U11_TASK2_WEIGHTS_PATH),
        'plate.weights_sha256': weights_sha256,
        'plate.image_size': U11_TASK2_PLATE_IMAGE_SIZE,
        'plate.confidence': U11_TASK2_PLATE_CONFIDENCE,
        'plate.processed_frame_count': int(frame_count),
    }
    actual = {
        'video_id': summary.get('video_id'),
        'frame_count': summary.get('frame_count'),
        **{f'pose.{key}': pose.get(key) for key in (
            'status', 'model', 'image_size', 'confidence',
            'keypoint_confidence', 'processed_frame_count',
        )},
        **{f'plate.{key}': plate.get(key) for key in (
            'status', 'model', 'weights_sha256', 'image_size',
            'confidence', 'processed_frame_count',
        )},
    }
    mismatches = [
        key for key, expected_value in expected.items()
        if actual.get(key) != expected_value
    ]
    if mismatches:
        raise ValueError(
            f'{video_id} 저장 추론 설정이 현재 설정과 다릅니다: '
            + ', '.join(mismatches)
            + '. RUN_U11_TASK2_INFERENCE=True로 다시 실행하세요.'
        )

## 7. 추가 9영상과 이전 실험 분리 확인

A/B/C 각 3명, 총 9명을 사용합니다. Task 1 pilot, B00, U10 학습·검증 사람과 `rule_blind` 영상은 제외합니다. `has_plate=unknown`은 억지로 yes로 바꾸지 않고 후보 검수에서 `no_plate` 또는 `target_missing`으로 기록합니다.

In [ ]:
task2_manifest = pd.read_csv(U11_TASK2_MANIFEST_PATH)
u10_video_split = pd.read_csv(U11_TASK2_U10_SPLIT_PATH)
excluded_video_ids = set(U11_VIDEO_IDS) | set(u10_video_split['video_id']) | {'b00_sn_f_01'}
excluded_person_ids = set(
    task2_manifest.loc[
        task2_manifest['video_id'].isin(excluded_video_ids), 'person_id'
    ].astype(str)
)
task2_rows = validate_u11_task2_video_selection(
    U11_TASK2_VIDEO_IDS,
    task2_manifest,
    excluded_person_ids=excluded_person_ids,
)
display(task2_rows[[
    'video_id', 'person_id', 'group', 'completion', 'frame_count',
    'has_plate', 'split_role',
]])

## 8. Task 2 후보 입력 생성

처음 한 번만 `RUN_U11_TASK2_INFERENCE=True`로 실행합니다. U10 Dataset V2 detector와 기존 YOLOv8n Pose를 그대로 사용하며, Pose는 주동작자 bbox 중심만 제공합니다. 완료 후 다시 `False`로 두면 저장 CSV를 재사용합니다.

In [ ]:
task2_inference_rows = []
task2_weights_sha256 = (
    sha256(U11_TASK2_WEIGHTS_PATH.read_bytes()).hexdigest()
    if U11_TASK2_WEIGHTS_PATH.exists() else None
)
if RUN_U11_TASK2_INFERENCE:
    if not U11_TASK2_WEIGHTS_PATH.exists():
        raise FileNotFoundError(f'Dataset V2 weight가 없습니다: {U11_TASK2_WEIGHTS_PATH}')
    verify_ultralytics_safe_load()
    from ultralytics import YOLO

    U11_TASK2_ROOT.mkdir(parents=True, exist_ok=True)
    for stale_path in (
        U11_TASK2_CANDIDATE_REVIEW_PATH, U11_TASK2_SELECTION_REVIEW_PATH,
        U11_TASK2_TARGET_PATH, U11_TASK2_EVALUATION_PATH,
        U11_TASK2_GROUP_SUMMARY_PATH, U11_TASK2_SUMMARY_PATH,
    ):
        stale_path.unlink(missing_ok=True)
    device_info = resolve_inference_device(U11_TASK2_REQUESTED_DEVICE)
    actual_device = device_info['actual_device']
    pose_model = YOLO(POSE_MODEL_NAME)
    plate_model = YOLO(str(U11_TASK2_WEIGHTS_PATH))

    for row in task2_rows.itertuples(index=False):
        video_id = str(row.video_id)
        video_path = PROJECT_ROOT / str(row.path)
        output_directory = U11_TASK2_INPUT_ROOT / video_id
        output_directory.mkdir(parents=True, exist_ok=True)
        for stale_name in (
            'pose_candidates.csv', 'pose_keypoints.csv',
            'plate_candidates.csv', 'plate_track.csv', 'run_summary.json',
        ):
            (output_directory / stale_name).unlink(missing_ok=True)
        summary = {
            'video_id': video_id, 'group': str(row.group),
            'frame_count': int(row.frame_count), 'device': device_info,
            'pose': {'status': 'not_run'}, 'plate': {'status': 'not_run'},
        }

        try:
            started = perf_counter()
            pose_candidates, pose_frame_count = infer_pose_video_candidates(
                pose_model, video_path=video_path, video_id=video_id,
                frame_count=int(row.frame_count), fps=float(row.fps),
                image_size=U11_TASK2_POSE_IMAGE_SIZE,
                confidence=U11_TASK2_POSE_CONFIDENCE, device=actual_device,
            )
            save_pose_candidate_cache(
                pose_candidates, output_directory / 'pose_candidates.csv'
            )
            primary = select_initial_primary_lifter(
                pose_candidates, frame_width=int(row.width)
            )
            if primary['candidate_id'] is None:
                raise ValueError('주동작자 자동 선택 실패: ' + str(primary['reason']))
            pose_keypoints = select_primary_lifter_track(
                pose_candidates, frame_count=int(row.frame_count),
                initial_frame_index=int(primary['frame_index']),
                initial_candidate_id=int(primary['candidate_id']),
                keypoint_confidence_threshold=U11_TASK2_KEYPOINT_CONFIDENCE,
                fps=float(row.fps),
            )
            save_selected_pose_keypoints(
                pose_keypoints, output_directory / 'pose_keypoints.csv'
            )
            summary['pose'] = {
                'status': 'complete', 'model': POSE_MODEL_NAME,
                'image_size': U11_TASK2_POSE_IMAGE_SIZE,
                'confidence': U11_TASK2_POSE_CONFIDENCE,
                'keypoint_confidence': U11_TASK2_KEYPOINT_CONFIDENCE,
                'processed_frame_count': pose_frame_count,
                'initial_selection': primary,
                'elapsed_seconds': perf_counter() - started,
            }
        except Exception as error:
            summary['pose'] = {
                'status': 'failed', 'reason': f'{type(error).__name__}: {error}'
            }

        try:
            started = perf_counter()
            plate_candidates, plate_frame_count = infer_plate_video_candidates(
                plate_model, video_path=video_path, video_id=video_id,
                frame_count=int(row.frame_count), fps=float(row.fps),
                image_width=int(row.width), image_height=int(row.height),
                image_size=U11_TASK2_PLATE_IMAGE_SIZE,
                confidence=U11_TASK2_PLATE_CONFIDENCE, device=actual_device,
            )
            save_plate_candidate_cache(
                plate_candidates, output_directory / 'plate_candidates.csv'
            )
            summary['plate'] = {
                'status': 'complete', 'model': str(U11_TASK2_WEIGHTS_PATH),
                'weights_sha256': task2_weights_sha256,
                'image_size': U11_TASK2_PLATE_IMAGE_SIZE,
                'confidence': U11_TASK2_PLATE_CONFIDENCE,
                'processed_frame_count': plate_frame_count,
                'candidate_row_count': len(plate_candidates),
                'elapsed_seconds': perf_counter() - started,
            }
        except Exception as error:
            summary['plate'] = {
                'status': 'failed', 'reason': f'{type(error).__name__}: {error}'
            }

        save_json_atomically(summary, output_directory / 'run_summary.json')
        task2_inference_rows.append({
            'video_id': video_id, 'pose_status': summary['pose']['status'],
            'plate_status': summary['plate']['status'],
        })
else:
    metadata_by_video = task2_rows.set_index('video_id')
    for video_id in U11_TASK2_VIDEO_IDS:
        summary_path = U11_TASK2_INPUT_ROOT / video_id / 'run_summary.json'
        if summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding='utf-8'))
            validate_task2_reused_summary(
                summary, video_id=video_id,
                frame_count=int(metadata_by_video.loc[video_id, 'frame_count']),
                weights_sha256=task2_weights_sha256,
            )
            task2_inference_rows.append({
                'video_id': video_id,
                'pose_status': summary['pose']['status'],
                'plate_status': summary['plate']['status'],
            })
        else:
            task2_inference_rows.append({
                'video_id': video_id, 'pose_status': 'not_run',
                'plate_status': 'not_run',
            })
display(pd.DataFrame(task2_inference_rows))

## 9. 자동 결과를 보기 전 수동 target 기록

회색 bbox와 candidate ID만 표시합니다. 주동작 원판 후보가 있으면 설정 셀에 `(frame_index, candidate_id)`, 원판이 있지만 올바른 후보가 없으면 `'target_missing'`, 영상에 원판이 없으면 `'no_plate'`를 입력합니다. 영상당 한 값만 기록하며 bbox를 다시 그리지 않습니다.

In [ ]:
task2_candidate_paths = {
    video_id: U11_TASK2_INPUT_ROOT / video_id / 'plate_candidates.csv'
    for video_id in U11_TASK2_VIDEO_IDS
}
missing_candidate_paths = [
    str(path) for path in task2_candidate_paths.values() if not path.exists()
]
if missing_candidate_paths:
    print('Task 2 후보 생성 대기:\n' + '\n'.join(missing_candidate_paths))
else:
    U11_TASK2_ROOT.mkdir(parents=True, exist_ok=True)
    metadata_by_video = task2_rows.set_index('video_id')
    figure, axes = plt.subplots(3, 3, figsize=(15, 21), squeeze=False)
    for axis, video_id in zip(axes.flat, U11_TASK2_VIDEO_IDS, strict=True):
        metadata = metadata_by_video.loc[video_id]
        candidates = pd.read_csv(task2_candidate_paths[video_id])
        frame_index = int(candidates['frame_index'].min()) if len(candidates) else 0
        first_candidates = candidates.loc[
            candidates['frame_index'].eq(frame_index)
        ]
        frame = read_video_frame(
            PROJECT_ROOT / str(metadata['path']), frame_index
        ).copy()
        for candidate in first_candidates.itertuples(index=False):
            start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
            end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
            cv2.rectangle(frame, start, end, (160, 160, 160), 4)
            cv2.putText(
                frame, f'id={int(candidate.candidate_id)}', start,
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (160, 160, 160), 2,
            )
        axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axis.set_title(
            f'{video_id} | frame={frame_index} | candidates={len(first_candidates)}'
        )
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(
        U11_TASK2_CANDIDATE_REVIEW_PATH, dpi=140, bbox_inches='tight'
    )
    plt.show()
    plt.close(figure)
    print(f'수동 target 검수 그림: {U11_TASK2_CANDIDATE_REVIEW_PATH}')

## 10. 자동 선택 비교와 참고 track

9개 수동 target을 모두 입력한 뒤 `RUN_U11_TASK2_EVALUATION=True`로 실행합니다. `target_missing`은 detector 누락으로 분리하고 selector 오선택 분모에 넣지 않습니다. track coverage는 프레임별 정답 bbox가 없으므로 정확도가 아닌 참고 지표입니다.

In [ ]:
task2_selection_evaluation = pd.DataFrame()
task2_group_summary = pd.DataFrame()
if not RUN_U11_TASK2_EVALUATION:
    pending_targets = [
        video_id for video_id, target in U11_TASK2_MANUAL_TARGETS.items()
        if target is None
    ]
    print(
        'Task 2 평가 대기: candidate review 후 수동 target을 입력하세요. '
        + ', '.join(pending_targets)
    )
else:
    required_task2_paths = []
    for video_id in U11_TASK2_VIDEO_IDS:
        required_task2_paths.extend([
            U11_TASK2_INPUT_ROOT / video_id / 'plate_candidates.csv',
            U11_TASK2_INPUT_ROOT / video_id / 'pose_keypoints.csv',
        ])
    missing_task2_paths = [
        str(path) for path in required_task2_paths if not path.exists()
    ]
    if missing_task2_paths:
        raise FileNotFoundError(
            'Task 2 평가 입력이 없습니다:\n' + '\n'.join(missing_task2_paths)
        )

    candidates_by_video = {
        video_id: pd.read_csv(
            U11_TASK2_INPUT_ROOT / video_id / 'plate_candidates.csv'
        )
        for video_id in U11_TASK2_VIDEO_IDS
    }
    all_task2_candidates = pd.concat(
        candidates_by_video.values(), ignore_index=True
    )
    manual_targets = validate_initial_plate_targets(
        U11_TASK2_MANUAL_TARGETS, video_ids=U11_TASK2_VIDEO_IDS,
        candidates=all_task2_candidates,
    )
    metadata_by_video = task2_rows.set_index('video_id')
    selection_rows = []
    total_frame_count = 0
    track_evaluated_frame_count = 0
    track_ok_frame_count = 0
    for video_id in U11_TASK2_VIDEO_IDS:
        metadata = metadata_by_video.loc[video_id]
        candidates = candidates_by_video[video_id]
        pose_keypoints = pd.read_csv(
            U11_TASK2_INPUT_ROOT / video_id / 'pose_keypoints.csv',
            usecols=['frame_index', 'bbox_center_x', 'bbox_center_y'],
        )
        lifter_centers = (
            pose_keypoints.dropna().drop_duplicates('frame_index')
            .rename(columns={
                'bbox_center_x': 'center_x_px',
                'bbox_center_y': 'center_y_px',
            })
        )
        selection = select_initial_plate_candidate(
            candidates, lifter_centers=lifter_centers,
            ambiguity_margin=U11_AMBIGUITY_MARGIN,
        )
        selection.update({'video_id': video_id, 'group': str(metadata['group'])})
        track_path = U11_TASK2_INPUT_ROOT / video_id / 'plate_track.csv'
        track_path.unlink(missing_ok=True)
        total_frame_count += int(metadata['frame_count'])
        if selection['status'] == 'selected':
            track = select_primary_plate_track(
                candidates, frame_count=int(metadata['frame_count']),
                initial_frame_index=int(selection['frame_index']),
                initial_candidate_id=int(selection['candidate_id']),
                lifter_centers=lifter_centers, fps=float(metadata['fps']),
            )
            save_plate_track(track, track_path)
            selection['track_ok_frame_count'] = int(track['status'].eq('ok').sum())
            track_evaluated_frame_count += int(metadata['frame_count'])
            track_ok_frame_count += int(selection['track_ok_frame_count'])
        else:
            selection['track_ok_frame_count'] = None
        selection_rows.append(selection)

    task2_selections = pd.DataFrame(selection_rows)
    task2_selection_evaluation, task2_summary = (
        evaluate_initial_plate_selections(task2_selections, manual_targets)
    )
    group_rows = []
    for group, group_selections in task2_selections.groupby('group', sort=True):
        group_video_ids = set(group_selections['video_id'])
        group_targets = manual_targets.loc[
            manual_targets['video_id'].isin(group_video_ids)
        ]
        _, group_summary = evaluate_initial_plate_selections(
            group_selections, group_targets
        )
        group_rows.append({'group': group, **group_summary})
    task2_group_summary = pd.DataFrame(group_rows)

    task2_summary.update({
        'status': 'complete',
        'decision': decide_initial_plate_selection_evaluation(
            task2_summary, task2_group_summary
        ),
        'track_ok_frame_count': track_ok_frame_count,
        'track_evaluated_frame_count': track_evaluated_frame_count,
        'track_not_evaluated_frame_count': (
            total_frame_count - track_evaluated_frame_count
        ),
        'total_frame_count': total_frame_count,
        'track_coverage_reference': (
            track_ok_frame_count / track_evaluated_frame_count
            if track_evaluated_frame_count else None
        ),
        'ambiguity_margin': U11_AMBIGUITY_MARGIN,
    })
    U11_TASK2_ROOT.mkdir(parents=True, exist_ok=True)
    manual_targets.to_csv(U11_TASK2_TARGET_PATH, index=False)
    task2_selection_evaluation.to_csv(U11_TASK2_EVALUATION_PATH, index=False)
    task2_group_summary.to_csv(U11_TASK2_GROUP_SUMMARY_PATH, index=False)
    save_json_atomically(task2_summary, U11_TASK2_SUMMARY_PATH)
    display(task2_selection_evaluation[[
        'video_id', 'group', 'status', 'reason', 'candidate_count',
        'target_status', 'matches_manual', 'selection_outcome',
        'track_ok_frame_count',
    ]])
    display(task2_group_summary)
    display(pd.DataFrame([task2_summary]))

In [ ]:
if task2_selection_evaluation.empty and U11_TASK2_EVALUATION_PATH.exists():
    task2_selection_evaluation = pd.read_csv(U11_TASK2_EVALUATION_PATH)
if task2_selection_evaluation.empty:
    print('Task 2 선택 결과 검수 대기: 먼저 평가를 실행하세요.')
else:
    metadata_by_video = task2_rows.set_index('video_id')
    figure, axes = plt.subplots(3, 3, figsize=(15, 21), squeeze=False)
    for axis, result in zip(
        axes.flat, task2_selection_evaluation.itertuples(index=False), strict=True
    ):
        metadata = metadata_by_video.loc[result.video_id]
        candidates = pd.read_csv(
            U11_TASK2_INPUT_ROOT / result.video_id / 'plate_candidates.csv'
        )
        frame_index = int(candidates['frame_index'].min()) if len(candidates) else 0
        first_candidates = candidates.loc[candidates['frame_index'].eq(frame_index)]
        frame = read_video_frame(
            PROJECT_ROOT / str(metadata['path']), frame_index
        ).copy()
        for candidate in first_candidates.itertuples(index=False):
            is_target = (
                result.target_status == 'selected'
                and int(candidate.candidate_id) == int(result.target_candidate_id)
            )
            is_auto = (
                result.status == 'selected'
                and int(candidate.candidate_id) == int(result.candidate_id)
            )
            color = (0, 255, 0) if is_target else (160, 160, 160)
            if is_auto and not is_target:
                color = (0, 0, 255)
            start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
            end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
            cv2.rectangle(frame, start, end, color, 4)
            cv2.putText(
                frame, f'id={int(candidate.candidate_id)}', start,
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2,
            )
        axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axis.set_title(f'{result.video_id} | {result.selection_outcome}')
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(
        U11_TASK2_SELECTION_REVIEW_PATH, dpi=140, bbox_inches='tight'
    )
    plt.show()
    plt.close(figure)
    print(f'자동 선택 검수 그림: {U11_TASK2_SELECTION_REVIEW_PATH}')

## 11. Task 2 판정

A/B/C마다 비교 가능한 원판 정답이 하나 이상 있고 오선택과 no-plate false positive가 0이면 통과합니다. 그룹별 근거가 없으면 `insufficient_evidence`, `confirmation_required`는 안전한 보류로 기록합니다. 실제 다중 후보가 없으면 거리 규칙을 더 복잡하게 만들지 않습니다. 실패가 확인될 때만 다음 실험에서 시간 연속성 근거를 한 축으로 추가합니다.